<a id="encrypted-array-operations-tutorial"></a>
# Encrypted Array Operations Tutorial

This notebook provides exhaustive testing and examples for every single function within the `arrays.py` module of the `concrete-fhe-toolkit`. We cover everything from simple element-wise math to complex cryptographic sorting and reductions.

When working with Fully Homomorphic Encryption (FHE), standard Python lists and NumPy arrays behave differently. We must use constant-time operations to avoid leaking information about the data size, positions, or conditions.

<a id="basic-array-math"></a>
## Basic Array Math
Functions covered: `array_add`, `array_sub`, `array_multiply`, `array_scale`, `array_sum`.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.arrays import array_add, array_sub, array_multiply, array_scale, array_sum

def test_array_math(a: list[int], b: list[int], factor: int):
    add_res = array_add(a, b)
    sub_res = array_sub(a, b)
    mul_res = array_multiply(a, b)
    scale_res = array_scale(a, factor)
    sum_res = array_sum(a)
    return add_res, sub_res, mul_res, scale_res, sum_res

# 1. Cleartext Execution
a, b, factor = [10, 20, 30], [1, 2, 3], 2
res_add, res_sub, res_mul, res_scale, res_sum = test_array_math(a, b, factor)

assert list(res_add) == [11, 22, 33]
assert list(res_sub) == [9, 18, 27]
assert list(res_mul) == [10, 40, 90]
assert list(res_scale) == [20, 40, 60]
assert res_sum == 60
print("Cleartext basic math passed!")

# 2. FHE Compilation
compiler = fhe.Compiler(test_array_math, {"a": "encrypted", "b": "encrypted", "factor": "clear"})
inputset = [([0, 0, 0], [1, 1, 1], 2), ([10, 20, 30], [1, 2, 3], 3)]
circuit = compiler.compile(inputset)

# 3. Encrypted Execution & Verification
enc_res = circuit.encrypt_run_decrypt(a, b, factor)
assert list(enc_res[0]) == list(res_add)
assert list(enc_res[1]) == list(res_sub)
assert list(enc_res[2]) == list(res_mul)
assert list(enc_res[3]) == list(res_scale)
assert enc_res[4] == res_sum
print("✅ Encrypted basic math passed all edge cases!")

<a id="array-manipulation"></a>
## Array Manipulation
Functions covered: `array_pad`, `array_slice`, `array_reverse`, `array_concat`, `array_cumsum`.

In [ ]:
from concrete_fhe_toolkit.arrays import array_pad, array_slice, array_reverse, array_concat, array_cumsum

def test_array_manip(a: list[int], b: list[int]):
    pad_res = array_pad(a, 5) # Pad to length 5
    slice_res = array_slice(a, 1, 3) # Slice indices 1 to 2
    rev_res = array_reverse(a)
    concat_res = array_concat(a, b)
    cumsum_res = array_cumsum(a)
    return pad_res, slice_res, rev_res, concat_res, cumsum_res

a, b = [10, 20, 30], [40, 50]
res_pad, res_slice, res_rev, res_concat, res_cumsum = test_array_manip(a, b)

assert list(res_pad) == [10, 20, 30, 0, 0]
assert list(res_slice) == [20, 30]
assert list(res_rev) == [30, 20, 10]
assert list(res_concat) == [10, 20, 30, 40, 50]
assert list(res_cumsum) == [10, 30, 60]
print("Cleartext manipulation passed!")

compiler = fhe.Compiler(test_array_manip, {"a": "encrypted", "b": "encrypted"})
inputset = [([0, 0, 0], [1, 1]), ([10, 20, 30], [40, 50])]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(a, b)
assert list(enc_res[0]) == list(res_pad)
assert list(enc_res[1]) == list(res_slice)
assert list(enc_res[2]) == list(res_rev)
assert list(enc_res[3]) == list(res_concat)
assert list(enc_res[4]) == list(res_cumsum)
print("✅ Encrypted manipulation passed!")

<a id="searching-querying-indexing"></a>
## Searching, Querying & Indexing
Functions covered: `array_contains`, `array_count`, `array_all_equal`, `array_index`, `array_index_of`, `array_set`.

> **Crucial FHE Note:** When accessing an array using a secret index (`array[secret_index]`), you cannot use standard indexing. `array_index` securely extracts the value without revealing the index, and `array_set` securely updates the array.

In [ ]:
from concrete_fhe_toolkit.arrays import (
    array_contains, array_count, array_all_equal, array_index, array_index_of, array_set
)

def test_array_query(arr: list[int], target: int, replace_val: int):
    contains = array_contains(arr, target)
    count = array_count(arr, target)
    is_eq = array_all_equal(arr, [1, 2, 2, 4])
    val_at_idx = array_index(arr, 2)
    idx_of = array_index_of(arr, target, missing_result=99)
    new_arr = array_set(arr, 1, replace_val)
    return contains, count, is_eq, val_at_idx, idx_of, new_arr

arr = [1, 2, 2, 4]
res_cont, res_cnt, res_eq, res_val, res_idx, res_set = test_array_query(arr, 2, 999)

assert res_cont == 1
assert res_cnt == 2
assert res_eq == 1
assert res_val == 2
assert res_idx == 1  # First occurrence of 2 is at index 1
assert list(res_set) == [1, 999, 2, 4]
print("Cleartext query passed!")

compiler = fhe.Compiler(test_array_query, {"arr": "encrypted", "target": "encrypted", "replace_val": "encrypted"})
inputset = [([0, 0, 0, 0], 0, 0), ([1, 2, 2, 4], 2, 999), ([1, 2, 2, 4], 9, 999)] # Target not found edge case!
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(arr, 2, 999)
assert enc_res[0] == res_cont
assert enc_res[1] == res_cnt
assert enc_res[2] == res_eq
assert enc_res[3] == res_val
assert enc_res[4] == res_idx
assert list(enc_res[5]) == res_set

# Edge case: Target not found!
enc_edge = circuit.encrypt_run_decrypt(arr, 9, 999)
assert enc_edge[0] == 0 # Contains false
assert enc_edge[1] == 0 # Count is 0
assert enc_edge[4] == 99 # missing_result triggered!
print("✅ Encrypted query passed standard and missing edge cases!")

<a id="minmax-reductions"></a>
## Min/Max Reductions
Functions covered: `compile_minimum`, `compile_maximum`, `compile_argmin`, `compile_argmax`.

These functions use tournament reduction algorithms to find extreme values. We test the `tie_break` edge cases to ensure stable deterministic behavior.

In [ ]:
from concrete_fhe_toolkit.arrays import compile_minimum, compile_maximum, compile_argmin, compile_argmax

arr = [10, 5, 20, 5]

circ_min = compile_minimum(size=4, min_value=5, max_value=20)
assert circ_min.encrypt_run_decrypt(arr) == 5

circ_max = compile_maximum(size=4, min_value=5, max_value=20)
assert circ_max.encrypt_run_decrypt(arr) == 20

# Test argmin with tie_break="first"
circ_argmin_first = compile_argmin(size=4, min_value=5, max_value=20, tie_break="first")
assert circ_argmin_first.encrypt_run_decrypt(arr) == 1  # First 5 is at index 1

# Test argmin with tie_break="last"
circ_argmin_last = compile_argmin(size=4, min_value=5, max_value=20, tie_break="last")
assert circ_argmin_last.encrypt_run_decrypt(arr) == 3   # Last 5 is at index 3

print("✅ Encrypted reductions passed tie-breaker edge cases!")

<a id="sorting-and-top-k"></a>
## Sorting and Top-K
Functions covered: `compile_sort`, `compile_compare_swap`, `compile_top_k`.

Sorting algorithms within FHE utilize *Bitonic Sort* circuits. We test ascending, descending, and getting the Top-K elements.

In [ ]:
from concrete_fhe_toolkit.arrays import compile_sort, compile_compare_swap, compile_top_k

# 1. Compare and Swap
circ_swap = compile_compare_swap(min_value=0, max_value=20)
assert tuple(circ_swap.encrypt_run_decrypt(15, 5)) == (5, 15)  # Swaps them!
assert tuple(circ_swap.encrypt_run_decrypt(2, 9)) == (2, 9)    # Leaves them!

# 2. Bitonic Sort (Size must be power of 2)
arr = [14, 2, 9, 5]
circ_sort_asc = compile_sort(size=4, min_value=0, max_value=15, descending=False)
assert list(circ_sort_asc.encrypt_run_decrypt(arr)) == [2, 5, 9, 14]

circ_sort_desc = compile_sort(size=4, min_value=0, max_value=15, descending=True)
assert list(circ_sort_desc.encrypt_run_decrypt(arr)) == [14, 9, 5, 2]

# 3. Top-K (get the 2 largest elements)
circ_top_k = compile_top_k(size=4, k=2, min_value=0, max_value=15, largest=True)
assert list(circ_top_k.encrypt_run_decrypt(arr)) == [14, 9]

print("✅ Encrypted sorting and Top-K passed all cases!")